# 18 — RKVS experiment: results exploration and visualisation

Loads outputs from **`17_rkvs_classifier.py`** (`results/rkvs_*.csv`, figures). Use this notebook for extra plots, tables, and thesis figures without re-running the full CV loop.

Sections **2–3** plot PR-AUC, ROC-AUC, precision, sensitivity, F1, and Youden J vs **K** (with focused SVM baselines). **4** shows per-fold histograms for a chosen classifier/K. **5** displays the impact point selection heatmap.

**Run first:** `python 17_rkvs_classifier.py` (from `transformation_experiment/` or with `data/` visible).

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

EXPERIMENT_DIR = Path.cwd() if Path("data").exists() else Path("transformation_experiment")
DATA_DIR = EXPERIMENT_DIR / "data"
RESULTS_DIR = EXPERIMENT_DIR / "results"
POC_DIR = Path("transformation_poc") if Path("transformation_poc").exists() else Path("..") / "transformation_poc"

print(RESULTS_DIR.resolve())

## 1. Summary tables

In [ ]:
df_sum = pd.read_csv(RESULTS_DIR / "rkvs_rskf_summary.csv")
df_long = pd.read_csv(RESULTS_DIR / "rkvs_rskf_metrics.csv")

print("Best K per classifier (by PR-AUC mean):")
best_by_clf = df_sum.loc[df_sum.groupby("classifier")["pr_auc_mean"].idxmax()]
display(best_by_clf[["K", "classifier", "pr_auc_mean", "pr_auc_std", "roc_auc_mean", "f1_mean"]])

print("\nFull summary (first 10 rows):")
display(df_sum.head(10))

## 2. PR-AUC vs K (from summary CSV)

In [ ]:
from scipy import stats as _stats

N_SPLITS = 50  # 10 repeats × 5 folds
_t_crit = _stats.t.ppf(0.975, N_SPLITS - 1)

K_ORDER = sorted(df_sum["K"].unique())

CLF_COLORS = {
    "lda": "#e41a1c",
    "qda": "#377eb8",
    "knn": "#4daf4a",
    "lr": "#984ea3",
    "svm": "#ff7f00",
}
CLF_LABELS = {
    "lda": "RKVS + LDA",
    "qda": "RKVS + QDA",
    "knn": "RKVS + k-NN",
    "lr": "RKVS + LR",
    "svm": "RKVS + SVM",
}

fig, ax = plt.subplots(figsize=(10, 5))
for clf in sorted(df_sum["classifier"].unique()):
    sub = df_sum[df_sum["classifier"] == clf].set_index("K")
    ks_available = [k for k in K_ORDER if k in sub.index]
    if not ks_available:
        continue
    m = sub.loc[ks_available, "pr_auc_mean"].values
    s = sub.loc[ks_available, "pr_auc_std"].values
    ci_half = _t_crit * s / np.sqrt(N_SPLITS)
    ax.plot(ks_available, m, "o-", color=CLF_COLORS[clf], label=CLF_LABELS[clf], ms=5, lw=2)
    ax.fill_between(ks_available, m - ci_half, m + ci_half, color=CLF_COLORS[clf], alpha=0.15)

# Baselines from focused_summary
df_base = pd.read_csv(RESULTS_DIR / "focused_summary.csv")

def best_svm(mask):
    r = df_base.loc[mask].sort_values("pr_auc_mean", ascending=False).iloc[0]
    return r["pr_auc_mean"], int(r["n_features"]), r["representation"]

m_xp, _, _ = best_svm((df_base["representation"] == "og_xp_110") & (df_base["classifier"] == "SVM_RBF"))
m_ch, k_ch, n_ch = best_svm(
    df_base["representation"].str.startswith("chebyshev")
    & df_base["representation"].str.endswith("_L2")
    & (df_base["classifier"] == "SVM_RBF")
)
m_leg, k_leg, n_leg = best_svm(
    df_base["representation"].str.startswith("legendre")
    & df_base["representation"].str.endswith("_L2")
    & (df_base["classifier"] == "SVM_RBF")
)

ax.axhline(m_xp, color="black", ls="--", lw=1.5, label=f"XP 110 + SVM ({m_xp:.3f})")
ax.axhline(m_ch, color="tab:blue", ls=":", lw=1.5, label=f"Best Chebyshev L2 + SVM (K={k_ch})")
ax.axhline(m_leg, color="tab:orange", ls=":", lw=1.5, label=f"Best Legendre L2 + SVM (K={k_leg})")

ax.set_xlabel("K (impact points selected)", fontsize=11)
ax.set_ylabel("PR-AUC (mean ± 95% CI)", fontsize=11)
ax.set_title("RKVS: PR-AUC vs K across classifiers", fontsize=12)
ax.legend(loc="lower right", fontsize=9)
ax.set_xticks(K_ORDER)
ax.grid(True, alpha=0.2)
fig.tight_layout()
plt.show()

## 3. Multi-metric grid (6 panels): ROC-AUC, precision, sensitivity, F1, Youden J

In [ ]:
df_base = pd.read_csv(RESULTS_DIR / "focused_summary.csv")

def baseline_svm_rows():
    def pick(mask):
        return df_base.loc[mask].sort_values("pr_auc_mean", ascending=False).iloc[0]
    m_xp = pick(
        (df_base["representation"] == "og_xp_110") & (df_base["classifier"] == "SVM_RBF")
    )
    m_ch = pick(
        df_base["representation"].str.startswith("chebyshev")
        & df_base["representation"].str.endswith("_L2")
        & (df_base["classifier"] == "SVM_RBF")
    )
    m_leg = pick(
        df_base["representation"].str.startswith("legendre")
        & df_base["representation"].str.endswith("_L2")
        & (df_base["classifier"] == "SVM_RBF")
    )
    k_ch, k_leg = int(m_ch["n_features"]), int(m_leg["n_features"])
    return m_xp, m_ch, m_leg, k_ch, k_leg

m_xp_b, m_ch_b, m_leg_b, k_ch_b, k_leg_b = baseline_svm_rows()

CLF_STYLE = [
    ("lda", "#e41a1c", "RKVS + LDA"),
    ("qda", "#377eb8", "RKVS + QDA"),
    ("knn", "#4daf4a", "RKVS + k-NN"),
    ("lr", "#984ea3", "RKVS + LR"),
    ("svm", "#ff7f00", "RKVS + SVM"),
]

def plot_metric_vs_k(ax, col: str, ylabel: str, title: str):
    """col = e.g. 'roc_auc' -> uses {col}_mean, {col}_std from df_sum."""
    mcol, scol = f"{col}_mean", f"{col}_std"
    for clf, color, lab in CLF_STYLE:
        sub = df_sum[df_sum["classifier"] == clf].set_index("K")
        ks_available = [k for k in K_ORDER if k in sub.index]
        if not ks_available:
            continue
        sub_plot = sub.loc[ks_available]
        ci_half = _t_crit * sub_plot[scol] / np.sqrt(N_SPLITS)
        ax.plot(ks_available, sub_plot[mcol], "o-", color=color, label=lab, ms=4)
        ax.fill_between(
            ks_available,
            sub_plot[mcol] - ci_half,
            sub_plot[mcol] + ci_half,
            color=color,
            alpha=0.15,
        )
    ax.axhline(
        m_xp_b[f"{col}_mean"],
        color="black",
        ls="--",
        lw=1,
        label=f"XP 110 + SVM ({m_xp_b[f'{col}_mean']:.3f})",
    )
    ax.axhline(
        m_ch_b[f"{col}_mean"],
        color="tab:blue",
        ls=":",
        lw=1,
        label=f"Best Chebyshev (K={k_ch_b})",
    )
    ax.axhline(
        m_leg_b[f"{col}_mean"],
        color="tab:orange",
        ls=":",
        lw=1,
        label=f"Best Legendre (K={k_leg_b})",
    )
    ax.set_xlabel("K (impact points)")
    ax.set_ylabel(ylabel)
    ax.set_title(title, fontsize=10)
    ax.legend(loc="lower right", fontsize=6)
    ax.set_xticks(K_ORDER)
    ax.grid(True, alpha=0.2)

fig, axes = plt.subplots(2, 3, figsize=(13, 8), sharex=True)
axes = axes.ravel()
panels = [
    ("roc_auc", "ROC-AUC (mean ± 95% CI)", "ROC-AUC vs K"),
    ("precision", "Precision (mean ± 95% CI)", "Precision vs K (Youden threshold)"),
    ("sensitivity", "Sensitivity (mean ± 95% CI)", "Sensitivity vs K"),
    ("f1", "F1 (mean ± 95% CI)", "F1 vs K"),
    ("youden_j", "Youden J (mean ± 95% CI)", "Youden J vs K"),
]
for ax, (col, ylab, ttl) in zip(axes[:5], panels):
    plot_metric_vs_k(ax, col, ylab, ttl)
axes[5].set_visible(False)
fig.suptitle("RKVS vs polynomial / XP baselines — additional metrics", y=1.00, fontsize=12)
fig.tight_layout()
plt.show()

## 4. Fold-level distributions (example: LDA at best K)

In [ ]:
clf_pick = "lda"
K_pick = int(df_sum[df_sum["classifier"] == clf_pick]["pr_auc_mean"].idxmax())
K_pick_val = df_sum.loc[K_pick, "K"]

sub = df_long[(df_long["classifier"] == clf_pick) & (df_long["K"] == K_pick_val)]

hist_cols = [
    ("pr_auc", "PR-AUC", "steelblue"),
    ("roc_auc", "ROC-AUC", "darkslategray"),
    ("precision", "Precision", "teal"),
    ("sensitivity", "Sensitivity", "coral"),
    ("f1", "F1", "mediumpurple"),
    ("youden_j", "Youden J", "olive"),
]
fig, axes = plt.subplots(2, 3, figsize=(12, 6), sharey=True)
axes = axes.ravel()
for ax, (col, lab, c) in zip(axes, hist_cols):
    ax.hist(sub[col].dropna(), bins=15, color=c, edgecolor="white", alpha=0.8)
    ax.set_xlabel(lab, fontsize=10)
    ax.set_ylabel("Count (folds)", fontsize=10)
    ax.set_title(lab, fontsize=10)
    ax.grid(True, alpha=0.2, axis="y")
fig.suptitle(f"{CLF_LABELS[clf_pick]}, K={K_pick_val} — fold distributions (RSKF)", fontsize=12, y=1.01)
fig.tight_layout()
plt.show()

## 5. Impact point selection frequency (heatmap)

In [ ]:
# Load and display the stability heatmap
img = plt.imread(RESULTS_DIR / "rkvs_impact_points_stability.png")
fig, ax = plt.subplots(figsize=(14, 4))
ax.imshow(img)
ax.axis("off")
fig.tight_layout()
plt.show()

print("Figure shows: RKVS impact point selection frequency across 50 RSKF folds.")
print("Vertical: top-K impact points (1–30). Horizontal: wavelength (336–1020 nm).")
print("Color intensity: fraction of 50 folds where each wavelength was selected.")

## 6. Class-mean spectra with impact points

In [ ]:
# Load and display the spectra with impact points
img2 = plt.imread(RESULTS_DIR / "rkvs_mean_spectra_with_impacts.png")
fig, ax = plt.subplots(figsize=(12, 5))
ax.imshow(img2)
ax.axis("off")
fig.tight_layout()
plt.show()

print("Figure shows: L2-normalised class-mean spectra (blue=single, orange=binary)")
print("with vertical red lines marking the most frequently selected impact points.")

## 7. Cross-experiment comparison table

In [ ]:
# Build a comparison table across RKVS, FPCA, ROCKET, and polynomial baselines
comparison_rows = []

# RKVS best by PR-AUC
rkvs_best_idx = df_sum["pr_auc_mean"].idxmax()
rkvs_best = df_sum.loc[rkvs_best_idx]
comparison_rows.append({
    "Method": f"RKVS + {rkvs_best['classifier'].upper()}",
    "K/J": int(rkvs_best["K"]),
    "PR-AUC": f"{rkvs_best['pr_auc_mean']:.4f} ± {rkvs_best['pr_auc_std']:.4f}",
    "ROC-AUC": f"{rkvs_best['roc_auc_mean']:.4f} ± {rkvs_best['roc_auc_std']:.4f}",
    "F1": f"{rkvs_best['f1_mean']:.4f} ± {rkvs_best['f1_std']:.4f}",
    "Sensitivity": f"{rkvs_best['sensitivity_mean']:.4f} ± {rkvs_best['sensitivity_std']:.4f}",
})

# FPCA best
if (RESULTS_DIR / "fpca_classifier_summary.csv").exists():
    df_fpca = pd.read_csv(RESULTS_DIR / "fpca_classifier_summary.csv")
    fpca_best_idx = df_fpca["pr_auc_mean"].idxmax()
    fpca_best = df_fpca.loc[fpca_best_idx]
    comparison_rows.append({
        "Method": f"FPCA + {fpca_best['classifier'].upper()}",
        "K/J": int(fpca_best["J"]),
        "PR-AUC": f"{fpca_best['pr_auc_mean']:.4f} ± {fpca_best['pr_auc_std']:.4f}",
        "ROC-AUC": f"{fpca_best['roc_auc_mean']:.4f} ± {fpca_best['roc_auc_std']:.4f}",
        "F1": f"{fpca_best['f1_mean']:.4f} ± {fpca_best['f1_std']:.4f}",
        "Sensitivity": f"{fpca_best['sensitivity_mean']:.4f} ± {fpca_best['sensitivity_std']:.4f}",
    })

# Polynomial baseline (Chebyshev)
ch_best = df_base.loc[
    (df_base["representation"].str.startswith("chebyshev"))
    & (df_base["representation"].str.endswith("_L2"))
    & (df_base["classifier"] == "SVM_RBF")
].sort_values("pr_auc_mean", ascending=False).iloc[0]
comparison_rows.append({
    "Method": f"Chebyshev L2 + SVM",
    "K/J": int(ch_best["n_features"]),
    "PR-AUC": f"{ch_best['pr_auc_mean']:.4f} ± {ch_best['pr_auc_std']:.4f}",
    "ROC-AUC": f"{ch_best['roc_auc_mean']:.4f} ± {ch_best['roc_auc_std']:.4f}",
    "F1": f"{ch_best['f1_mean']:.4f} ± {ch_best['f1_std']:.4f}",
    "Sensitivity": f"{ch_best['sensitivity_mean']:.4f} ± {ch_best['sensitivity_std']:.4f}",
})

# Native XP baseline
xp_best = df_base[df_base["representation"] == "og_xp_110"].sort_values("pr_auc_mean", ascending=False).iloc[0]
comparison_rows.append({
    "Method": f"XP 110-d + {xp_best['classifier']}",
    "K/J": 110,
    "PR-AUC": f"{xp_best['pr_auc_mean']:.4f} ± {xp_best['pr_auc_std']:.4f}",
    "ROC-AUC": f"{xp_best['roc_auc_mean']:.4f} ± {xp_best['roc_auc_std']:.4f}",
    "F1": f"{xp_best['f1_mean']:.4f} ± {xp_best['f1_std']:.4f}",
    "Sensitivity": f"{xp_best['sensitivity_mean']:.4f} ± {xp_best['sensitivity_std']:.4f}",
})

df_comp = pd.DataFrame(comparison_rows)
display(df_comp)